In [73]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings("ignore")


# Pre-processing

In [2]:
DATA_PATH ='data/train-house-prices-advanced-regression-techniques.csv'
house_df = pd.read_csv(DATA_PATH)
house_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


Dropping Feature that contains missing value more than 50/100. So that reducing noise and negative effecting to train model, besides helps retain important and reliable features

In [ ]:
 # droping values more than 50/100
house_df = house_df.drop(["Id","Alley","PoolQC","Fence","MiscFeature"], axis=1)

In [23]:
def get_categorical_features(df):
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    total = []
    for col in cat_cols:
        print(f"{col}: {df[col].nunique()} unique values")
        print(f"Values: {df[col].unique()}")
        print("-" * 40)
        total.append(len(df[col].unique()))
    total = np.sum(total)
    print('Total Unique: ',total )

    return cat_cols.tolist()
cat_features = get_categorical_features(house_df)

MSZoning: 5 unique values
Values: <ArrowStringArray>
['RL', 'RM', 'C (all)', 'FV', 'RH']
Length: 5, dtype: str
----------------------------------------
Street: 2 unique values
Values: <ArrowStringArray>
['Pave', 'Grvl']
Length: 2, dtype: str
----------------------------------------
LotShape: 4 unique values
Values: <ArrowStringArray>
['Reg', 'IR1', 'IR2', 'IR3']
Length: 4, dtype: str
----------------------------------------
LandContour: 4 unique values
Values: <ArrowStringArray>
['Lvl', 'Bnk', 'Low', 'HLS']
Length: 4, dtype: str
----------------------------------------
Utilities: 2 unique values
Values: <ArrowStringArray>
['AllPub', 'NoSeWa']
Length: 2, dtype: str
----------------------------------------
LotConfig: 5 unique values
Values: <ArrowStringArray>
['Inside', 'FR2', 'Corner', 'CulDSac', 'FR3']
Length: 5, dtype: str
----------------------------------------
LandSlope: 3 unique values
Values: <ArrowStringArray>
['Gtl', 'Mod', 'Sev']
Length: 3, dtype: str
-------------------------

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_27588\1211634446.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object', 'category']).columns


Although One-Hot Encoding increases the number of features to approximately 250, we prefer it over Ordinal Encoding because it preserves categorical distinctions without introducing an artificial order between categories.

In [50]:
train_df, test_df = train_test_split(
    house_df, test_size=0.25, random_state=42)

y_train = train_df["SalePrice"].values
y_test = test_df["SalePrice"].values

train_df = train_df.drop(["SalePrice"],axis = 1)
test_df = test_df.drop(["SalePrice"],axis = 1)


One-hot and fill missing value 

In [60]:
#getting num and cat feature names
num_cols = [col for col in train_df.columns if train_df[col].dtype in ["float64","int64"]]
cat_cols = [col for col in train_df.columns if train_df[col].dtype not in ["float64","int64"]]

#fill none cat features
train_df[cat_cols] = train_df[cat_cols].fillna("none")
test_df[cat_cols] = test_df[cat_cols].fillna("none")

#fill value num_cols
imputer = SimpleImputer()
train_df[num_cols] = imputer.fit_transform(train_df[num_cols])
test_df[num_cols] = imputer.transform(test_df[num_cols])

#Encode
#Option handle_unknown for ignoring categorical that not appear in training_dataset
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(train_df[cat_cols])

encoded_cols = list(encoder.get_feature_names_out(cat_cols))

train_df[encoded_cols] = encoder.fit_transform(train_df[cat_cols])
test_df[encoded_cols] = encoder.transform(test_df[cat_cols])


Normalization

In [72]:
scaler = MinMaxScaler()
train_num_features = scaler.fit_transform(train_df[num_cols])
test_num_features = scaler.transform(test_df[num_cols])

In [71]:
#Concat num and cat features
X_train = np.concatenate((train_num_features,train_df[encoded_cols]), axis = 1 )
X_test  = np.concatenate((test_num_features,test_df[encoded_cols]), axis = 1 )


Using LinearRegression, Ridge, Lasso with Linear Feature

In [92]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, HuberRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# Tạo danh sách model
models = {
    'Linear Regression: ': LinearRegression(),
      'Ridge: ': Ridge(),
      'Lasso: ': Lasso(),
      'Huber Regressor': HuberRegressor(),
      'MLP: ':TransformedTargetRegressor(
    regressor=MLPRegressor(hidden_layer_sizes=(64, 32), alpha=0.01,
                           early_stopping=True, max_iter=1000, random_state=42),
    transformer=StandardScaler() )
}

# Khởi tạo list lưu kết quả
train_rmse_results = []
test_rmse_results = []
train_r2_results = []
test_r2_results = []
model_names = []

# Huấn luyện và tính metric
for name, model in models.items():
    regressor = model.fit(X_train,y_train)

    # Dự đoán
    y_train_pred = regressor.predict(X_train)
    y_test_pred =  regressor.predict(X_test)

    # Tính RMSE
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    # Tính R²
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    # Lưu kết quả
    model_names.append(name)
    train_rmse_results.append(train_rmse)
    test_rmse_results.append(test_rmse)
    train_r2_results.append(train_r2)
    test_r2_results.append(test_r2)

# Tạo DataFrame tổng hợp
df_results = pd.DataFrame({
    "Model": model_names,
    "Train_RMSE": train_rmse_results,
    "Test_RMSE": test_rmse_results,
    "Train_R2": train_r2_results,
    "Test_R2": test_r2_results
}).sort_values(by="Test_R2", ascending=False)

df_results


,Model,Train_RMSE,Test_RMSE,Train_R2,Test_R2
2,Lasso:,20032.668558,26362.510069,0.933902,0.900792
3,Huber Regressor,27704.431975,27890.849043,0.873583,0.888956
0,Linear Regression:,20029.220254,28137.140208,0.933925,0.886986
1,Ridge:,22488.286583,28586.775808,0.916705,0.883345
4,MLP:,8339.651144,31820.560031,0.988545,0.855460


In [ ]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

mlp = TransformedTargetRegressor(
    regressor=MLPRegressor(hidden_layer_sizes=(64, 32), alpha=0.01,
                           early_stopping=True, max_iter=1000, random_state=42),
    transformer=StandardScaler(),
)
mlp.fit(X_train, y_train)
mlp_results = []
for split, X, y in [("Train", X_train, y_train), ("Test", X_test, y_test)]:
    prediction = mlp.predict(X)
    mlp_results.append({"Split": split, "RMSE": np.sqrt(mean_squared_error(y, prediction)),
                        "R2": r2_score(y, prediction)})
pd.DataFrame(mlp_results).set_index("Split")

,RMSE,R2
Split,,
Train,8339.651144,0.988545
Test,31819.903253,0.855466


In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly_features = PolynomialFeatures(
    degree=2, interaction_only=True, include_bias=False
)

train_poly_features = poly_features.fit_transform(train_df[num_cols])
test_poly_features = poly_features.transform(test_df[num_cols])

scaler = MinMaxScaler()
train_poly_features = scaler.fit_transform(train_poly_features)
test_poly_features = scaler.transform(test_poly_features)

X_train_poly = np.hstack([train_poly_features, train_df[encoded_cols].values])
X_test_poly = np.hstack([test_poly_features, test_df[encoded_cols].values])

# making dictionary of models
models = {
    "Linear Regression: ": LinearRegression(),
    "Ridge: ": Ridge(),
    "Lasso: ": Lasso(),
    "Huber Regressor": HuberRegressor(),
    "MLP: ": TransformedTargetRegressor(
        regressor=MLPRegressor(
            hidden_layer_sizes=(64, 32), alpha=0.01,
            early_stopping=True, max_iter=1000, random_state=42,
        ),
        transformer=StandardScaler(),
    ),
}

# lists to store results
r2_results = []
rmse_results = []
train_rmse_results = []
train_r2_results = []
model_names = []

# training and evaluating each model
for name, model in models.items():
    regressor = model.fit(X_train_poly, y_train)

    # predict
    y_train_pred = regressor.predict(X_train_poly)
    y_test_pred = regressor.predict(X_test_poly)

    # metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    # store
    model_names.append(name)
    train_rmse_results.append(train_rmse)
    rmse_results.append(test_rmse)
    train_r2_results.append(train_r2)
    r2_results.append(test_r2)

# create dataframe
df_results = pd.DataFrame({
    "Model": model_names,
    "Train_RMSE": train_rmse_results,
    "Test_RMSE": rmse_results,
    "Train_R2": train_r2_results,
    "Test_R2": r2_results,
}).sort_values(by="Test_R2", ascending=False)

df_results